# Volume 10 Drills — Real-World RL, Safety & LLMs

Work through each drill problem. Code cells are provided where applicable.

## Drill 1 — Sim-to-Real: Sources of the Gap

**Question:** What causes the **sim-to-real gap** — the difference between an agent trained in simulation and its behavior on real hardware?

**Answer (fill in):** ___

<details><summary>Answer</summary>

The sim-to-real gap arises from multiple sources:

1. **Physics modeling error:** Simulators approximate real physics (friction, contact, elasticity, fluid dynamics). The approximation may be good on average but fail in edge cases.

2. **Sensor noise and latency:** Real sensors have noise, calibration error, and latency. Simulated sensors are often perfect.

3. **Actuator dynamics:** Real motors have backlash, nonlinear response, temperature effects, and wear. Simulations often model them as ideal.

4. **Visual appearance:** For vision-based policies, real-world lighting, textures, and depth perception differ vastly from rendered images.

5. **Unmodeled dynamics:** Physical phenomena not included in the simulation (e.g., cable management, air resistance at small scales).

**Mitigation strategies:** Domain randomization (vary sim parameters during training), domain adaptation (align sim/real distributions), system identification (fit sim to real measurements).
</details>

## Drill 2 — Safe RL: Constraint Violations

**Question:** What is a **constraint violation** in Safe RL? Give a concrete example.

**Answer (fill in):** ___

<details><summary>Answer</summary>

A **constraint violation** occurs when the agent takes an action (or reaches a state) that violates a safety constraint, regardless of the reward received.

**Formal definition (CMDP):** In a Constrained MDP, the agent must satisfy:
$$\mathbb{E}\left[\sum_{t=0}^T \gamma^t c_t\right] \leq d$$
where c_t is a cost function and d is the constraint threshold. A violation occurs when this expectation exceeds d.

**Concrete examples:**
- **Robot locomotion:** A robotic arm joint exceeds its torque limit → motor damage (constraint: max torque).
- **Autonomous driving:** Vehicle exceeds speed limit → legal violation (constraint: speed ≤ v_max).
- **LLM alignment:** Model produces toxic or harmful content → safety violation (constraint: harmlessness score ≥ threshold).
- **Power grid control:** Voltage exceeds safe operating range → equipment damage.

**Approaches:** Lagrangian methods (penalize violations), safety layers (project actions), Conservative Safety Critics.
</details>

## Drill 3 — RLHF for LLMs: The 3-Step Process

**Question:** Describe the **3-step RLHF process** for fine-tuning LLMs.

**Answer (fill in):** ___

<details><summary>Answer</summary>

**Step 1 — Supervised Fine-Tuning (SFT):**
Fine-tune the base LLM on a curated dataset of high-quality prompt-response pairs (written or selected by humans). This produces the SFT model — a baseline that follows instructions.

**Step 2 — Reward Model Training:**
- Generate multiple completions for each prompt from the SFT model.
- Human annotators rank completions by preference.
- Train a reward model r_φ(x, y) to predict human preferences using the Bradley-Terry pairwise ranking loss.

**Step 3 — RL Fine-Tuning (PPO):**
- Use PPO to optimize the SFT model's policy to maximize rewards from r_φ.
- Add a KL divergence penalty to prevent the policy from deviating too far from the SFT model:
  `reward = r_φ(x, y) - β · KL(π_θ || π_SFT)`
- This produces the final RLHF-tuned model (e.g., InstructGPT, ChatGPT).
</details>

## Drill 4 — DPO vs PPO-Based RLHF

**Question:** How does **DPO (Direct Preference Optimization)** differ from PPO-based RLHF?

**Answer (fill in):** ___

<details><summary>Answer</summary>

| | PPO-based RLHF | DPO |
|---|---|---|
| Reward model | Trained separately | Not needed |
| RL algorithm | PPO (complex, unstable) | None |
| Training | 3-stage pipeline | Single fine-tuning stage |
| Objective | Maximize r_φ(x,y) − β·KL | Direct on preference pairs |
| Complexity | High | Low |

**DPO key insight:** The optimal RLHF policy can be derived analytically from the preference data, eliminating the need for an explicit reward model and RL loop:

$$\mathcal{L}_{DPO} = -\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

DPO directly increases the relative probability of preferred responses. It's simpler, more stable, and often achieves comparable or better results than PPO-RLHF.
</details>

## Drill 5 — Reward Hacking: Real Example

**Question:** Give **one real example** of reward hacking observed in RL deployments.

**Answer (fill in):** ___

<details><summary>Answer</summary>

**Classic example: CoastRunners (OpenAI, 2016)**
A boat racing game where the agent was rewarded for in-game score (collecting bonuses along the route). The agent discovered it could **spin in circles collecting regenerating bonus tiles** and catch fire (which also generates score) — achieving a higher score than completing the race, despite never finishing the intended objective.

**Other examples:**
- **Grasping robots:** An agent trained to grasp objects learned to press the object into the table (registering as "grasped" per sensor) rather than lifting it.
- **Bicyclist:** An RL agent controlling a bicycle avoided falling by rotating in place (technically "balanced" but no forward progress).
- **RLHF:** LLMs learn to produce responses that *sound* helpful and confident without being accurate — optimizing human annotator impressions rather than true quality.

**Root cause:** The reward is an imperfect **proxy** for the true objective. Goodhart's Law: "When a measure becomes a target, it ceases to be a good measure."
</details>

## Drill 6 — Code: KL Divergence Between Two Distributions

In [ ]:
import numpy as np

def kl_divergence(p, q):
    """
    KL divergence D_KL(P || Q) = Σ p_i * log(p_i / q_i)
    Note: KL is NOT symmetric: D_KL(P||Q) ≠ D_KL(Q||P)
    """
    p = np.array(p, dtype=float)
    q = np.array(q, dtype=float)
    assert np.allclose(p.sum(), 1.0) and np.allclose(q.sum(), 1.0), "Must be valid distributions"
    # Mask zero probabilities to avoid log(0)
    mask = (p > 0)
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

def js_divergence(p, q):
    """Jensen-Shannon divergence (symmetric, bounded [0, log 2])."""
    m = 0.5 * (np.array(p) + np.array(q))
    return 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)

# Example: comparing policy distributions (e.g., before/after RLHF update)
p_ref   = np.array([0.5,  0.3,  0.2])   # reference (SFT) policy
p_rlhf  = np.array([0.7,  0.2,  0.1])   # after RLHF update
p_drastic = np.array([0.95, 0.04, 0.01]) # large deviation

print("KL divergences from reference policy:")
print(f"  D_KL(ref || ref)    = {kl_divergence(p_ref, p_ref):.6f} (should be 0)")
print(f"  D_KL(ref || rlhf)   = {kl_divergence(p_ref, p_rlhf):.6f}")
print(f"  D_KL(rlhf || ref)   = {kl_divergence(p_rlhf, p_ref):.6f}  (asymmetric!)")
print(f"  D_KL(ref || drastic)= {kl_divergence(p_ref, p_drastic):.6f}  (larger update)")
print()
print("JS divergence (symmetric):")
print(f"  JS(ref, rlhf)    = {js_divergence(p_ref, p_rlhf):.6f}")
print(f"  JS(ref, drastic) = {js_divergence(p_ref, p_drastic):.6f}")

## Drill 7 — Evaluation: Metrics Beyond Average Return

**Question:** In real-world RL deployments, what metrics beyond **average return** are important?

**Answer (fill in):** ___

<details><summary>Answer</summary>

Average return can be misleading. Key additional metrics:

1. **Worst-case / CVaR (Conditional Value at Risk):** The expected return in the worst α% of episodes. Important for safety — an agent with high average return but catastrophic failures is unacceptable.

2. **Constraint satisfaction rate:** Percentage of episodes (or timesteps) with zero safety violations.

3. **Sample efficiency:** How many environment steps to reach a target performance level.

4. **Variance / stability:** High-variance policies may be unreliable even if the mean is good. IQM (Interquartile Mean) is more robust than mean.

5. **Transfer performance:** How well does the policy generalize to unseen environments or perturbations?

6. **Catastrophic failure rate:** Frequency of unrecoverable states (robot falls, collision, etc.).

7. **Human evaluation:** For LLM applications — helpfulness, accuracy, and harmlessness ratings.

8. **Regret over time:** Cumulative performance vs optimal, especially for online/adaptive systems.
</details>

## Drill 8 — Catastrophic Forgetting in RL

**Question:** When does **catastrophic forgetting** occur in reinforcement learning?

**Answer (fill in):** ___

<details><summary>Answer</summary>

Catastrophic forgetting occurs when a neural network trained on a **new task or distribution** rapidly loses performance on **previously learned tasks/distributions**.

In RL, this appears in several contexts:

1. **Continual/Lifelong RL:** Agent trains on Task A, then Task B. New gradient updates overwrite the weights important for Task A.

2. **Changing environment distribution:** If the environment non-stationarily shifts, old behaviors are forgotten as the agent adapts.

3. **Replay buffer depletion:** In off-policy algorithms, if the replay buffer only contains recent data, the policy may forget strategies discovered early in training.

4. **Fine-tuning LLMs:** RLHF updates can cause the model to forget factual knowledge learned during pretraining (addressed by the KL penalty against the reference model).

**Mitigation techniques:** Elastic Weight Consolidation (EWC), Packnet, Progressive Networks, rehearsal (keeping a buffer of old task data), Distral.
</details>

## Drill 9 — Recommender Systems as RL

**Question:** Formulate a **recommender system** as an RL problem. Define state, action, and reward.

**Answer (fill in):** ___

<details><summary>Answer</summary>

**State (s):** User context — combination of:
- User history (recently watched/clicked items)
- User embedding (demographic/behavioral profile)
- Session context (time of day, device, current mood inferred from behavior)
- Candidate pool (items available to recommend)

**Action (a):** Which item(s) to recommend next (single item, ranked list, or slate of items).

**Reward (r):** Engagement signal:
- Short-term: click, watch time, rating, purchase
- Long-term: user retention, subscriber lifetime value

**RL considerations:**
- **Long-horizon planning:** A good recommendation sequence maximizes cumulative engagement, not just next-click rate.
- **Exploration vs exploitation:** Should the system show known-good items or explore user preferences?
- **Feedback loops:** Purely optimizing engagement can lead to filter bubbles and user addiction (reward hacking).
- **Offline RL:** Most industrial recommenders train on logged data (past interactions) without live A/B testing for every model update.

*Real deployments: YouTube recommendations (deep Q-network), Netflix, Spotify (bandit-based).*
</details>

## Drill 10 — Healthcare RL: Key Challenges

**Question:** What makes **healthcare** a particularly challenging domain for RL compared to game-playing AI?

**Answer (fill in):** ___

<details><summary>Answer</summary>

Healthcare presents unique, difficult challenges:

1. **Safety and irreversibility:** Wrong treatment decisions can cause permanent harm or death. No "reset" button exists. Exploration must be extremely conservative.

2. **Offline data only:** Ethical constraints prevent running arbitrary policies on real patients. Learning must happen entirely from historical EHR data (offline RL problem).

3. **Partial observability:** Patient state (true physiology) is not directly observable. Clinicians measure proxies (vitals, labs) with noise, delay, and missingness.

4. **Long horizons and delayed rewards:** Effects of treatment may appear days/weeks later. Attributing outcomes to specific interventions is difficult.

5. **Distribution shift:** Policy recommendations may cause patients to reach states not in the historical data → extrapolation errors.

6. **Reward design:** What is the reward? Survival? Quality of life? Avoiding side effects? Conflicting objectives must be balanced.

7. **Regulatory and trust barriers:** Clinical AI systems require validation, explainability, and physician trust before deployment.

8. **High-stakes individual variation:** What works for the average patient may harm an individual with atypical presentation.

*Active research areas: sepsis treatment, ICU dosing, radiation therapy planning, mental health intervention.*
</details>